In [ ]:
import pandas as pd
import pathlib

# 1. Load the Excel file ONCE (returns a dictionary of DataFrames)
file_loc = r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\all_products_divided.xlsx'
all_sheets = pd.read_excel(file_loc, sheet_name=None)

def calculate_stats(df):
    """Processes a single DataFrame and returns its cluster statistics."""
    # Ensure columns exist before processing
    if 'Cluster' not in df.columns or 'Price' not in df.columns:
        return None
        
    # Clean data
    df = df.dropna(subset=['Cluster', 'Price'])
    
    # 1. Generate descriptive stats
    stats = df.groupby('Cluster')['Price'].describe()
    
    # 2. Add CV and IQR
    stats['cv_%'] = (stats['std'] / stats['mean']) * 100
    stats['IQR'] = stats['75%'] - stats['25%']

    # 3. Reorder and convert to dictionary
    cols = ['count', 'mean', 'std', 'cv_%', 'min', '25%', '50%', '75%', 'max', 'IQR']
    stats = stats[cols]
    
    return stats.to_dict(orient='index')

# 2. Loop through all sheets and build the cat_stats dictionary
cat_stats = {}
final_stats = {}

for category in all_sheets.keys():
    sheet_df = all_sheets[category]
    stats_dict = calculate_stats(sheet_df)
    
    if stats_dict:
        cat_stats[category] = stats_dict

# --- Example: How to access the data ---
# To see the stats for 'baby_kids' as a DataFrame:
for cat in cat_stats.keys():
    final_stats[cat] = pd.DataFrame(cat_stats[cat]).T
# Define the output path
output_path = r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\final_stats_summary.xlsx'

# Use ExcelWriter to save multiple sheets into one file
with pd.ExcelWriter(output_path) as writer:
    for sheet_name, df in final_stats.items():
        # Sheet names have a 31-character limit in Excel
        clean_name = str(sheet_name)[:31]
        df.to_excel(writer, sheet_name=clean_name)

print(f"Success! Final stats saved to: {output_path}")

Success! Final stats saved to: C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\final_stats_summary.xlsx


In [10]:
# 1. Combine all DataFrames in the dictionary into one
# The 'keys' argument uses the dictionary keys to create a MultiIndex
master_df = pd.concat(final_stats, names=['category', 'Cluster'])

# 2. Reset the index to turn 'category' and 'Cluster' into regular columns
master_df = master_df.reset_index()

# Display the result
master_df.to_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\analysis.csv', index=False)

In [11]:
import pandas as pd
import pathlib

def get_cluster_stats(file_path, category='baby_kids'):
    path = pathlib.Path(file_path)
    if not path.exists():
        print(f"Error: The file {file_path} was not found.")
        return None

    try:
        df = pd.read_csv(path, usecols=['Cluster Name', 'Price'])
        df = df.dropna(subset=['Cluster Name', 'Price'])

        # 1. Generate standard descriptive stats
        stats = df.groupby('Cluster Name')['Price'].describe()
        
        # 2. Calculate CV (Standard Deviation Percentage)
        # Formula: (std / mean) * 100
        stats['cv_%'] = (stats['std'] / stats['mean']) * 100
        
        # 3. Calculate IQR
        stats['IQR'] = stats['75%'] - stats['25%']

        # 4. Reorder columns to place cv_% between std and min
        cols = ['count', 'mean', 'std', 'cv_%', 'min', '25%', '50%', '75%', 'max', 'IQR']
        stats = stats[cols]

        # Convert to dictionary
        master_dict = stats.to_dict(orient='index')
        return master_dict

    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None

# Usage
file_loc = r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\baby_kids.csv'
master_dict = get_cluster_stats(file_loc)
# Displaying the result
baby_df = pd.DataFrame(master_dict).T
baby_df

An unexpected error occurred: 'utf-8' codec can't decode byte 0xa0 in position 1436: invalid start byte


""


In [14]:
import pandas as pd

# Define the file path
file_path = r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\all_products_divided.xlsx'

# 1. Load the Excel file (None loads all sheets as a dictionary)
excel_file = pd.read_excel(file_path, sheet_name=None)

all_sheets = []

for sheet_name, df in excel_file.items():
    # 2. Create the 'category' column using the sheet name
    df['category'] = sheet_name
    all_sheets.append(df)

# 3. Merge all dataframes into one master dataframe
master_df = pd.concat(all_sheets, ignore_index=True)

# Optional: Save the master file
# master_df.to_excel('master_products.xlsx', index=False)

print(f"Merged {len(all_sheets)} sheets into a master dataframe with {master_df.shape[0]} rows.")
master_df.drop(columns=['Canonical Text', 'SKUs', 'Cluster #']).to_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\all_products_final.csv',index=False)

Merged 10 sheets into a master dataframe with 997 rows.


In [7]:
import pandas as pd

pd.read_csv(r'C:\Users\moham\OneDrive\Desktop\WebScraping\legacy_app\output\all\all_products_final.csv').drop('coarse_category', axis=1)

,Product,Price,Handle,Inventory Management,SKU,Product URL,Scraped at,Description,Brand,Cluster,category
0,Astronauts - Swaddle Towel sets,995,astronauts-swaddle-towel-sets,shopify,672b9440b801c,https://hebalinens.com/products/astronauts-swa...,2/11/2026 15:44,Snuggle your little space explorer with our As...,Heba Linens,Swaddle Towel,baby_kids
1,Baby animals - Swaddle Towel sets,995,baby-animals-swaddle-towel-sets,shopify,672b9440b0b7a,https://hebalinens.com/products/baby-animals-s...,2/11/2026 15:44,"Soft and cuddly, this baby swaddle towel set w...",Heba Linens,Swaddle Towel,baby_kids
2,Hapiona - Swaddle Towel set,995,butterflies-swaddle-towel-sets,shopify,672b965ed8cc0,https://hebalinens.com/products/butterflies-sw...,2/11/2026 15:46,Experience joy with every cuddle! Our Butterfl...,Heba Linens,Swaddle Towel,baby_kids
3,Stars - Swaddle Towel sets,995,swaddle-towel-sets,shopify,672b977e50947,https://hebalinens.com/products/swaddle-towel-...,2/11/2026 15:47,Ensure the sweetest dream for your little one ...,Heba Linens,Swaddle Towel,baby_kids
4,Ballerina - Duvet set,6395,ballerina-duvet-set,shopify,672b9442c140d,https://hebalinens.com/products/ballerina-duve...,2/11/2026 15:44,Let your dreams take flight with this Ballerin...,Heba Linens,Duvet,baby_kids
...,...,...,...,...,...,...,...,...,...,...,...
992,Rattan Tray,490,ratan-tray,shopify,NaN,https://eg.lilly-home.com//products/ratan-tray,2/11/2026 16:44,handmade wooden rattan tray,Lilly Home,Other,home_decor
993,Retro circluar glass vase,480,retro-circluar-glass-vase,shopify,NaN,https://eg.lilly-home.com//products/retro-circ...,2/11/2026 16:44,Material : Handmade glass,Lilly Home,Glass Vase,home_decor
994,Retro cup glass vase,420,retro-glass-vase,shopify,NaN,https://eg.lilly-home.com//products/retro-glas...,2/11/2026 16:44,Material : Handmade glass,Lilly Home,Glass Vase,home_decor
995,Retro cylinder glass vase,420,retro-cylind-glass-vase,shopify,NaN,https://eg.lilly-home.com//products/retro-cyli...,2/11/2026 16:44,Material : Handmade glass,Lilly Home,Glass Vase,home_decor
